# Module 21 – KNN Case Study on Diabetes

**Objective:** Explore a diabetes dataset, preprocess it, train a K-Nearest Neighbors (KNN) classifier, evaluate it, and explain the results.

**Dataset:** Use the Pima Indians Diabetes CSV commonly named `diabetes.csv`, downloaded from your instructor/Kaggle source. It should contain an `Outcome` column (0/1). Upload the CSV to the same folder as this notebook or to your Colab session.

**Educational use only:** This model is not a medical diagnostic tool and must not be used for healthcare decisions.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
print("Libraries imported successfully")

## 2. Load the dataset
The code looks for `diabetes.csv`. In Colab, it will ask you to upload the file if it is not found.

In [ ]:
dataset_path = Path("diabetes.csv")
if not dataset_path.exists():
    try:
        from google.colab import files
        print("Upload diabetes.csv")
        uploaded = files.upload()
        if uploaded:
            dataset_path = Path(next(iter(uploaded.keys())))
    except ImportError:
        pass
if not dataset_path.exists():
    raise FileNotFoundError("Place diabetes.csv beside this notebook, or upload it in Colab.")
df = pd.read_csv(dataset_path)
print("Shape:", df.shape)
display(df.head())

## 3. Explore the data
Inspect dimensions, columns, data types, summary statistics, missing values, duplicates, and target balance.

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()
display(df.describe().T)
print("Missing values:")
display(df.isnull().sum().to_frame("Missing values"))
print("Duplicate rows:", df.duplicated().sum())

## 4. Data cleaning and preprocessing
For the classic Pima dataset, zero values in Glucose, BloodPressure, SkinThickness, Insulin, and BMI are commonly treated as missing measurement placeholders. We leave zero in Pregnancies and Outcome unchanged. The median imputer and scaler are fitted inside a pipeline using training data only, preventing data leakage.

In [ ]:
target_column = "Outcome"
if target_column not in df.columns:
    raise ValueError(f"Expected '{target_column}'. Found: {df.columns.tolist()}")
df = df.drop_duplicates().copy()
X = df.drop(columns=[target_column]).copy()
y = df[target_column].copy()
if y.isnull().any():
    raise ValueError("Target contains missing values; inspect the dataset first.")
zero_as_missing = [c for c in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"] if c in X.columns]
for col in zero_as_missing:
    X.loc[X[col] == 0, col] = np.nan
print("Zero-coded measurement columns:", zero_as_missing)
print("Target counts:")
display(y.value_counts().sort_index().to_frame("Count"))
print("Feature missing values:")
display(X.isnull().sum().to_frame("Missing values"))

## 5. Split into training and testing sets
We use 80% for training and 20% for testing. Stratification preserves the target class proportions approximately.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## 6. Train KNN and select hyperparameters
KNN relies on distances, so standardization is important. We compare values of k, distance weighting, and Manhattan/Euclidean distance using 5-fold cross-validation on the training set. The test set is reserved for final evaluation.

In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 19, 21],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}
search = GridSearchCV(pipeline, param_grid, scoring="f1", cv=5, n_jobs=-1, refit=True)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best parameters:", search.best_params_)
print(f"Best mean CV F1: {search.best_score_:.4f}")

## 7. Evaluate on the test set
Accuracy measures overall correctness. Precision measures the reliability of positive predictions. Recall measures how many actual positive cases are found. F1 combines precision and recall. The confusion matrix displays true/false predictions.

In [ ]:
y_pred = best_model.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Predicted 0", "Predicted 1"], yticklabels=["Actual 0", "Actual 1"])
plt.title("KNN Confusion Matrix"); plt.xlabel("Prediction"); plt.ylabel("Actual"); plt.tight_layout(); plt.show()
if hasattr(best_model, "predict_proba") and y_test.nunique() == 2:
    probabilities = best_model.predict_proba(X_test)[:, 1]
    print(f"ROC-AUC  : {roc_auc_score(y_test, probabilities):.4f}")
    RocCurveDisplay.from_predictions(y_test, probabilities)
    plt.title("KNN ROC Curve"); plt.grid(alpha=0.3); plt.show()

## 8. Compare values of k
The plot shows the best cross-validation F1-score for each tested neighborhood size, across the other settings.

In [ ]:
results = pd.DataFrame(search.cv_results_)
k_summary = (results.assign(k=results["param_knn__n_neighbors"].astype(int))
             .groupby("k", as_index=False)["mean_test_score"].max().sort_values("k"))
display(k_summary.rename(columns={"mean_test_score": "Best CV F1 for k"}))
plt.figure(figsize=(8, 4))
plt.plot(k_summary["k"], k_summary["mean_test_score"], marker="o")
plt.title("Cross-Validation F1 Score by K"); plt.xlabel("Number of Neighbors (k)"); plt.ylabel("Best Mean CV F1")
plt.xticks(k_summary["k"]); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 9. Short analysis of results
After running the notebook, complete this paragraph using your actual output:

> The KNN model was trained after median imputation and feature standardization. Cross-validation selected **[insert best parameters]**. On the held-out test set, accuracy was **[value]**, precision was **[value]**, recall was **[value]**, and F1-score was **[value]**. The confusion matrix shows **[describe correct predictions and false positives/negatives]**. These results describe performance on this dataset and split only; they do not establish clinical reliability.

**Limitation:** Dataset quality, class imbalance, train/test split, and hyperparameter choices affect results. This is a learning exercise, not a diagnostic system.

## 10. Optional: save a cleaned copy for inspection

In [ ]:
cleaned_df = X.copy()
cleaned_df[target_column] = y
cleaned_df.to_csv("diabetes_cleaned_for_analysis.csv", index=False)
print("Saved diabetes_cleaned_for_analysis.csv")

## Submission checklist
- [ ] Run all cells from top to bottom
- [ ] Confirm charts and metric outputs appear
- [ ] Fill in the analysis paragraph using your results
- [ ] Save as `KNN_Diabetes.ipynb`
- [ ] Upload/open it in Google Colab
- [ ] Share with **Anyone with the link – Viewer**
- [ ] Copy the actual Colab link into your submission

Do not invent a Colab link; create it from your own Google account.